# Notebook 01 — SQL Server → MinIO (Landing Zone / CSV)

Extrai todas as tabelas do banco **SeguroDB** (SQL Server) e grava cada uma como arquivo CSV no bucket `landing-zone` do **MinIO**.

**Fluxo:**
```
SQL Server (SeguroDB)  →  pyodbc/pandas  →  boto3  →  MinIO s3://landing-zone/<tabela>.csv
```

> Este notebook **não usa Apache Spark** — a extração é feita com pyodbc e o upload com boto3 para simplicidade e velocidade.

## 1. Importações e Configuração

In [ ]:
import io
import os
import pyodbc
import boto3
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from botocore.client import Config

load_dotenv(find_dotenv())

# SQL Server
SQL_SERVER   = os.getenv('SQL_SERVER',   'localhost')
SQL_PORT     = os.getenv('SQL_PORT',     '1433')
SQL_USER     = os.getenv('SQL_USER',     'sa')
SQL_PASSWORD = os.getenv('SQL_PASSWORD', 'SqlServer@2022!')
SQL_DATABASE = os.getenv('SQL_DATABASE', 'SeguroDB')

# MinIO
MINIO_ENDPOINT       = os.getenv('MINIO_ENDPOINT',       'http://localhost:9020')
MINIO_ACCESS_KEY     = os.getenv('MINIO_ACCESS_KEY',     'minioadmin')
MINIO_SECRET_KEY     = os.getenv('MINIO_SECRET_KEY',     'minioadmin')
MINIO_LANDING_BUCKET = os.getenv('MINIO_LANDING_BUCKET', 'landing-zone')

print('Configuração carregada.')
print(f'  SQL Server : {SQL_SERVER}:{SQL_PORT}/{SQL_DATABASE}')
print(f'  MinIO      : {MINIO_ENDPOINT} → bucket [{MINIO_LANDING_BUCKET}]')

## 2. Conectar ao SQL Server

In [ ]:
conn_str = (
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={SQL_SERVER},{SQL_PORT};'
    f'DATABASE={SQL_DATABASE};'
    f'UID={SQL_USER};PWD={SQL_PASSWORD};'
    f'TrustServerCertificate=yes;'
)

conn = pyodbc.connect(conn_str)
print(f'Conectado ao SQL Server — banco: {SQL_DATABASE}')

## 3. Listar tabelas disponíveis no SeguroDB

In [ ]:
query_tabelas = """
    SELECT TABLE_NAME
    FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_TYPE = 'BASE TABLE'
    ORDER BY TABLE_NAME
"""

df_tabelas = pd.read_sql(query_tabelas, conn)
tabelas = df_tabelas['TABLE_NAME'].tolist()

print(f'{len(tabelas)} tabelas encontradas:')
for t in tabelas:
    print(f'  - {t}')

## 4. Criar bucket `landing-zone` no MinIO (se não existir)

In [ ]:
s3 = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1',
)

existing_buckets = [b['Name'] for b in s3.list_buckets().get('Buckets', [])]

if MINIO_LANDING_BUCKET not in existing_buckets:
    s3.create_bucket(Bucket=MINIO_LANDING_BUCKET)
    print(f'Bucket [{MINIO_LANDING_BUCKET}] criado.')
else:
    print(f'Bucket [{MINIO_LANDING_BUCKET}] já existe.')

## 5. Extrair tabelas e enviar como CSV para o MinIO

In [ ]:
resumo = []

for tabela in tabelas:
    # Extrai do SQL Server
    df = pd.read_sql(f'SELECT * FROM dbo.{tabela}', conn)
    n_rows = len(df)
    
    # Serializa para CSV em memória
    buffer = io.BytesIO()
    df.to_csv(buffer, index=False, encoding='utf-8')
    buffer.seek(0)
    
    # Envia para MinIO
    object_key = f'{tabela}.csv'
    s3.put_object(
        Bucket=MINIO_LANDING_BUCKET,
        Key=object_key,
        Body=buffer.getvalue(),
        ContentType='text/csv',
    )
    
    size_kb = len(buffer.getvalue()) / 1024
    resumo.append({'tabela': tabela, 'registros': n_rows, 'tamanho_kb': round(size_kb, 2)})
    print(f'  [{MINIO_LANDING_BUCKET}/{object_key}]  {n_rows} linhas  ({size_kb:.1f} KB)')

conn.close()
print('\nExtração concluída. Conexão SQL Server encerrada.')

## 6. Resumo da extração

In [ ]:
df_resumo = pd.DataFrame(resumo)
df_resumo['destino'] = df_resumo['tabela'].apply(lambda t: f's3a://{MINIO_LANDING_BUCKET}/{t}.csv')
print(df_resumo.to_string(index=False))
print(f'\nTotal: {df_resumo["registros"].sum()} registros  |  {df_resumo["tamanho_kb"].sum():.1f} KB')

## 7. Listar arquivos no bucket landing-zone

In [ ]:
response = s3.list_objects_v2(Bucket=MINIO_LANDING_BUCKET)
print(f'Arquivos no bucket [{MINIO_LANDING_BUCKET}]:')
for obj in response.get('Contents', []):
    print(f'  {obj["Key"]:30s}  {obj["Size"]/1024:7.1f} KB  {obj["LastModified"].strftime("%Y-%m-%d %H:%M:%S")}')